## Master

In [2]:
from pyscf import gto, scf, dft, fci, mcscf
import matplotlib.pyplot as plt
from functools import reduce
import numpy as np
from pyqake.lib.Transpile import PySCFTranspiler
from pyqake.operator.observable import Dipole
from pyqake.lib import MO2AO

master_dir  = '***'

## $H_2$

In [26]:
###===================================================<<< Settings

system      = 'H2'
dist_lst    = np.around(np.arange(0.5, 2.01, 0.1), 3)
N_state     = 3
active_e    = (1, 1)
active_o    = 2

###===================================================<<< Calculation

energy_CAS  = np.zeros((len(dist_lst), N_state))
dipole_CAS  = np.zeros((len(dist_lst), N_state, N_state, 3))

for ind, dist in enumerate(dist_lst):
    print(f"\n>>>>>> {dist} AA Start")
    # HF
    mol, mf_chk         = scf.chkfile.load_scf(f"{master_dir}/Data/{system}/Chkfile/scf_{dist}")
    mf_hf               = scf.RHF(mol)
    mf_hf.__dict__.update(mf_chk)
    Trans               = PySCFTranspiler(mf_hf, active_o, active_e)
    dip_ints            = mol.intor('cint1e_r_sph', comp=3)

    # CASCI
    mc_mo               = mf_hf.mo_coeff
    mc                  = mcscf.CASCI(mf_hf, active_o, active_e)
    mc.fcisolver        = fci.direct_spin0.FCI(mol)
    mc.fcisolver.nroots = N_state
    mc.kernel(mc_mo)
    energy_CAS[ind]     = mc.e_tot

    # CASCI RDM & DC
    def cas_make_rdm(ci_id):
        t_dm = mc.fcisolver.trans_rdm1(mc.ci[ci_id], mc.ci[ci_id], mc.ncas, mc.nelecas)
        return t_dm

    for ex in range(N_state):
        D_cas_active    = cas_make_rdm(ex)
        D_cas_full      = MO2AO.restore_full_space(D_cas_active, Trans)
        D_cas_AO        = MO2AO.run(D_cas_full, mf_hf.mo_coeff)
        np.save(f"{master_dir}/Data/{system}/Density/CASCI_{ex}_{dist}", D_cas_full)
    print("CASCI RDM Done")

    # CASCI Dipole
    dip_frozen      = Dipole.gen_core_dipole(classic_object = PySCFTranspiler(mf_hf, active_o, active_e))
    def makedip(ci_id1, ci_id2):
        t_dm1       = mc.fcisolver.trans_rdm1(mc.ci[ci_id1], mc.ci[ci_id2], mc.ncas, mc.nelecas)
        orbcas      = mc_mo[:,mc.ncore:mc.ncore+mc.ncas]
        t_dm1_ao    = reduce(np.dot, (orbcas, t_dm1, orbcas.T))
        return np.einsum('xij,ji->x', dip_ints, t_dm1_ao)

    for i in range(N_state):
        for j in range(N_state):
            dip                         = makedip(i,j)
            dipole_CAS[ind, i, j, :]    = dip + (dip_frozen if i == j else 0.0)
    print("CASCI Dipole Done")

    np.save(f"{master_dir}/Data/{system}/Energy/{dist}_CAS", energy_CAS[ind])
    np.save(f"{master_dir}/Data/{system}/Dipole/{dist}_CAS", dipole_CAS[ind])


>>>>>> 0.5 AA Start
=====================< PySCF Transpiler >=====================
      Multiplicity : 1
 # Spatial Orbital : 2 (Active Space Applied)
       # Electrons : Alpha(1) | Beta(1)
            # Atom : 2
            Core E : 1.0584 Hartree

WARN: Mulitple states found in CASCI solver. First state is used to compute the Fock matrix and natural orbitals in active space.

CASCI state   0  E = -1.05515979447063  E(CI) = -2.11351421631063  S^2 = 0.0000000
CASCI state   1  E = 0.267000340959025  E(CI) = -0.791354080880975  S^2 = 0.0000000
CASCI state   2  E = 1.30148574734635  E(CI) = 0.243131325506351  S^2 = 0.0000000


CASCI RDM Done
=====================< PySCF Transpiler >=====================
      Multiplicity : 1
 # Spatial Orbital : 2 (Active Space Applied)
       # Electrons : Alpha(1) | Beta(1)
            # Atom : 2
            Core E : 1.0584 Hartree
CASCI Dipole Done

>>>>>> 0.6 AA Start
=====================< PySCF Transpiler >=====================
      Multiplicity : 1
 # Spatial Orbital : 2 (Active Space Applied)
       # Electrons : Alpha(1) | Beta(1)
            # Atom : 2
            Core E : 0.8820 Hartree

WARN: Mulitple states found in CASCI solver. First state is used to compute the Fock matrix and natural orbitals in active space.

CASCI state   0  E = -1.11628600686954  E(CI) = -1.99824802506954  S^2 = 0.0000000
CASCI state   1  E = 0.0365011952359020  E(CI) = -0.845460822964098  S^2 = 0.0000000
CASCI state   2  E = 0.890084668679598  E(CI) = 0.00812265047959815  S^2 = 0.0000000
CASCI RDM Done
=====================< PySCF Transpiler >=====================
      Multiplicity : 

## $H_2$ Large

In [27]:
###===================================================<<< Settings

system      = 'H2_Large'
dist_lst    = np.around(np.arange(0.5, 2.01, 0.1), 3)
N_state     = 3
active_e    = (1, 1)
active_o    = 4

###===================================================<<< Calculation

energy_CAS  = np.zeros((len(dist_lst), N_state))
dipole_CAS  = np.zeros((len(dist_lst), N_state, N_state, 3))

for ind, dist in enumerate(dist_lst):
    print(f"\n>>>>>> {dist} AA Start")
    # HF
    mol, mf_chk         = scf.chkfile.load_scf(f"{master_dir}/Data/{system}/Chkfile/scf_{dist}")
    mf_hf               = scf.RHF(mol)
    mf_hf.__dict__.update(mf_chk)
    Trans               = PySCFTranspiler(mf_hf, active_o, active_e)
    dip_ints            = mol.intor('cint1e_r_sph', comp=3)

    # CASCI
    mc_mo               = mf_hf.mo_coeff
    mc                  = mcscf.CASCI(mf_hf, active_o, active_e)
    mc.fcisolver        = fci.direct_spin0.FCI(mol)
    mc.fcisolver.nroots = N_state
    mc.kernel(mc_mo)
    energy_CAS[ind]     = mc.e_tot

    # CASCI RDM & DC
    def cas_make_rdm(ci_id):
        t_dm = mc.fcisolver.trans_rdm1(mc.ci[ci_id], mc.ci[ci_id], mc.ncas, mc.nelecas)
        return t_dm

    for ex in range(N_state):
        D_cas_active    = cas_make_rdm(ex)
        D_cas_full      = MO2AO.restore_full_space(D_cas_active, Trans)
        D_cas_AO        = MO2AO.run(D_cas_full, mf_hf.mo_coeff)
        np.save(f"{master_dir}/Data/{system}/Density/CASCI_{ex}_{dist}", D_cas_full)
    print("CASCI RDM Done")

    # CASCI Dipole
    dip_frozen      = Dipole.gen_core_dipole(classic_object = PySCFTranspiler(mf_hf, active_o, active_e))
    def makedip(ci_id1, ci_id2):
        t_dm1       = mc.fcisolver.trans_rdm1(mc.ci[ci_id1], mc.ci[ci_id2], mc.ncas, mc.nelecas)
        orbcas      = mc_mo[:,mc.ncore:mc.ncore+mc.ncas]
        t_dm1_ao    = reduce(np.dot, (orbcas, t_dm1, orbcas.T))
        return np.einsum('xij,ji->x', dip_ints, t_dm1_ao)

    for i in range(N_state):
        for j in range(N_state):
            dip                         = makedip(i,j)
            dipole_CAS[ind, i, j, :]    = dip + (dip_frozen if i == j else 0.0)
    print("CASCI Dipole Done")

    np.save(f"{master_dir}/Data/{system}/Energy/{dist}_CAS", energy_CAS[ind])
    np.save(f"{master_dir}/Data/{system}/Dipole/{dist}_CAS", dipole_CAS[ind])


>>>>>> 0.5 AA Start
=====================< PySCF Transpiler >=====================
      Multiplicity : 1
 # Spatial Orbital : 4 (Active Space Applied)
       # Electrons : Alpha(1) | Beta(1)
            # Atom : 2
            Core E : 1.0584 Hartree

WARN: Mulitple states found in CASCI solver. First state is used to compute the Fock matrix and natural orbitals in active space.

CASCI state   0  E = -1.07786389657449  E(CI) = -2.13621831841449  S^2 = 0.0000000
CASCI state   1  E = -0.431341259956064  E(CI) = -1.48969568179606  S^2 = 0.0000000
CASCI state   2  E = -0.0550981798416816  E(CI) = -1.11345260168168  S^2 = 0.0000000
CASCI RDM Done
=====================< PySCF Transpiler >=====================
      Multiplicity : 1
 # Spatial Orbital : 4 (Active Space Applied)
       # Electrons : Alpha(1) | Beta(1)
            # Atom : 2
            Core E : 1.0584 Hartree
CASCI Dipole Done

>>>>>> 0.6 AA Start
=====================< PySCF Transpiler >=====================
      Multiplici

## $H_2$ Huge

In [28]:
###===================================================<<< Settings

system      = 'H2_Huge'
dist_lst    = np.around(np.arange(0.5, 2.01, 0.1), 3)
N_state     = 3
active_e    = (1, 1)
active_o    = 5

###===================================================<<< Calculation

energy_CAS  = np.zeros((len(dist_lst), N_state))
dipole_CAS  = np.zeros((len(dist_lst), N_state, N_state, 3))

for ind, dist in enumerate(dist_lst):
    print(f"\n>>>>>> {dist} AA Start")
    # HF
    mol, mf_chk         = scf.chkfile.load_scf(f"{master_dir}/Data/{system}/Chkfile/scf_{dist}")
    mf_hf               = scf.RHF(mol)
    mf_hf.__dict__.update(mf_chk)
    Trans               = PySCFTranspiler(mf_hf, active_o, active_e)
    dip_ints            = mol.intor('cint1e_r_sph', comp=3)

    # CASCI
    mc_mo               = mf_hf.mo_coeff
    mc                  = mcscf.CASCI(mf_hf, active_o, active_e)
    mc.fcisolver        = fci.direct_spin0.FCI(mol)
    mc.fcisolver.nroots = N_state
    mc.kernel(mc_mo)
    energy_CAS[ind]     = mc.e_tot

    # CASCI RDM & DC
    def cas_make_rdm(ci_id):
        t_dm = mc.fcisolver.trans_rdm1(mc.ci[ci_id], mc.ci[ci_id], mc.ncas, mc.nelecas)
        return t_dm

    for ex in range(N_state):
        D_cas_active    = cas_make_rdm(ex)
        D_cas_full      = MO2AO.restore_full_space(D_cas_active, Trans)
        D_cas_AO        = MO2AO.run(D_cas_full, mf_hf.mo_coeff)
        np.save(f"{master_dir}/Data/{system}/Density/CASCI_{ex}_{dist}", D_cas_full)
    print("CASCI RDM Done")

    # CASCI Dipole
    dip_frozen      = Dipole.gen_core_dipole(classic_object = PySCFTranspiler(mf_hf, active_o, active_e))
    def makedip(ci_id1, ci_id2):
        t_dm1       = mc.fcisolver.trans_rdm1(mc.ci[ci_id1], mc.ci[ci_id2], mc.ncas, mc.nelecas)
        orbcas      = mc_mo[:,mc.ncore:mc.ncore+mc.ncas]
        t_dm1_ao    = reduce(np.dot, (orbcas, t_dm1, orbcas.T))
        return np.einsum('xij,ji->x', dip_ints, t_dm1_ao)

    for i in range(N_state):
        for j in range(N_state):
            dip                         = makedip(i,j)
            dipole_CAS[ind, i, j, :]    = dip + (dip_frozen if i == j else 0.0)
    print("CASCI Dipole Done")

    np.save(f"{master_dir}/Data/{system}/Energy/{dist}_CAS", energy_CAS[ind])
    np.save(f"{master_dir}/Data/{system}/Dipole/{dist}_CAS", dipole_CAS[ind])


>>>>>> 0.5 AA Start
=====================< PySCF Transpiler >=====================
      Multiplicity : 1
 # Spatial Orbital : 5 (Active Space Applied)
       # Electrons : Alpha(1) | Beta(1)
            # Atom : 2
            Core E : 1.0584 Hartree

WARN: Mulitple states found in CASCI solver. First state is used to compute the Fock matrix and natural orbitals in active space.

CASCI state   0  E = -1.07653796478379  E(CI) = -2.13489238662379  S^2 = 0.0000000
CASCI state   1  E = -0.496187492691873  E(CI) = -1.55454191453187  S^2 = 0.0000000
CASCI state   2  E = -0.396659424476125  E(CI) = -1.45501384631613  S^2 = 0.0000000
CASCI RDM Done
=====================< PySCF Transpiler >=====================
      Multiplicity : 1
 # Spatial Orbital : 5 (Active Space Applied)
       # Electrons : Alpha(1) | Beta(1)
            # Atom : 2
            Core E : 1.0584 Hartree
CASCI Dipole Done

>>>>>> 0.6 AA Start
=====================< PySCF Transpiler >=====================
      Multiplicit

## $LiH$

In [29]:
###===================================================<<< Settings

system      = 'LiH'
dist_lst    = np.around(np.arange(1.0, 2.01, 0.1), 3)
N_state     = 4
active_e    = (1, 1)
active_o    = 5

###===================================================<<< Calculation

energy_CAS  = np.zeros((len(dist_lst), N_state))
dipole_CAS  = np.zeros((len(dist_lst), N_state, N_state, 3))

for ind, dist in enumerate(dist_lst):
    print(f"\n>>>>>> {dist} AA Start")
    # HF
    mol, mf_chk         = scf.chkfile.load_scf(f"{master_dir}/Data/{system}/Chkfile/scf_{dist}")
    mf_hf               = scf.RHF(mol)
    mf_hf.__dict__.update(mf_chk)
    Trans               = PySCFTranspiler(mf_hf, active_o, active_e)
    dip_ints            = mol.intor('cint1e_r_sph', comp=3)

    # CASCI
    mc_mo               = mf_hf.mo_coeff
    mc                  = mcscf.CASCI(mf_hf, active_o, active_e)
    mc.fcisolver        = fci.direct_spin0.FCI(mol)
    mc.fcisolver.nroots = N_state
    mc.kernel(mc_mo)
    energy_CAS[ind]     = mc.e_tot

    # CASCI RDM & DC
    def cas_make_rdm(ci_id):
        t_dm = mc.fcisolver.trans_rdm1(mc.ci[ci_id], mc.ci[ci_id], mc.ncas, mc.nelecas)
        return t_dm

    for ex in range(N_state):
        D_cas_active    = cas_make_rdm(ex)
        D_cas_full      = MO2AO.restore_full_space(D_cas_active, Trans)
        D_cas_AO        = MO2AO.run(D_cas_full, mf_hf.mo_coeff)
        np.save(f"{master_dir}/Data/{system}/Density/CASCI_{ex}_{dist}", D_cas_full)
    print("CASCI RDM Done")

    # CASCI Dipole
    dip_frozen      = Dipole.gen_core_dipole(classic_object = PySCFTranspiler(mf_hf, active_o, active_e))
    def makedip(ci_id1, ci_id2):
        t_dm1       = mc.fcisolver.trans_rdm1(mc.ci[ci_id1], mc.ci[ci_id2], mc.ncas, mc.nelecas)
        orbcas      = mc_mo[:,mc.ncore:mc.ncore+mc.ncas]
        t_dm1_ao    = reduce(np.dot, (orbcas, t_dm1, orbcas.T))
        return np.einsum('xij,ji->x', dip_ints, t_dm1_ao)

    for i in range(N_state):
        for j in range(N_state):
            dip                         = makedip(i,j)
            dipole_CAS[ind, i, j, :]    = dip + (dip_frozen if i == j else 0.0)
    print("CASCI Dipole Done")

    np.save(f"{master_dir}/Data/{system}/Energy/{dist}_CAS", energy_CAS[ind])
    np.save(f"{master_dir}/Data/{system}/Dipole/{dist}_CAS", dipole_CAS[ind])


>>>>>> 1.0 AA Start
=====================< PySCF Transpiler >=====================
      Multiplicity : 1
 # Spatial Orbital : 5 (Active Space Applied)
       # Electrons : Alpha(1) | Beta(1)
            # Atom : 2
            Core E : -6.6098 Hartree

WARN: Mulitple states found in CASCI solver. First state is used to compute the Fock matrix and natural orbitals in active space.

CASCI state   0  E = -7.78402132044816  E(CI) = -1.17423654931062  S^2 = 0.0000000
CASCI state   1  E = -7.64380962842344  E(CI) = -1.03402485728590  S^2 = 0.0000000
CASCI state   2  E = -7.58517539501878  E(CI) = -0.975390623881242  S^2 = 0.0000000
CASCI state   3  E = -7.58517539501878  E(CI) = -0.975390623881242  S^2 = 0.0000000
CASCI RDM Done
=====================< PySCF Transpiler >=====================
      Multiplicity : 1
 # Spatial Orbital : 5 (Active Space Applied)
       # Electrons : Alpha(1) | Beta(1)
            # Atom : 2
            Core E : -6.6098 Hartree
CASCI Dipole Done

>>>>>> 1.1 AA S

## $LiH$ Large

In [30]:
###===================================================<<< Settings

system      = 'LiH_Large'
dist_lst    = np.around(np.arange(1.0, 2.01, 0.1), 3)
N_state     = 4
active_e    = (1, 1)
active_o    = 5

###===================================================<<< Calculation

energy_CAS  = np.zeros((len(dist_lst), N_state))
dipole_CAS  = np.zeros((len(dist_lst), N_state, N_state, 3))

for ind, dist in enumerate(dist_lst):
    print(f"\n>>>>>> {dist} AA Start")
    # HF
    mol, mf_chk         = scf.chkfile.load_scf(f"{master_dir}/Data/{system}/Chkfile/scf_{dist}")
    mf_hf               = scf.RHF(mol)
    mf_hf.__dict__.update(mf_chk)
    Trans               = PySCFTranspiler(mf_hf, active_o, active_e)
    dip_ints            = mol.intor('cint1e_r_sph', comp=3)

    # CASCI
    mc_mo               = mf_hf.mo_coeff
    mc                  = mcscf.CASCI(mf_hf, active_o, active_e)
    mc.fcisolver        = fci.direct_spin0.FCI(mol)
    mc.fcisolver.nroots = N_state
    mc.kernel(mc_mo)
    energy_CAS[ind]     = mc.e_tot

    # CASCI RDM & DC
    def cas_make_rdm(ci_id):
        t_dm = mc.fcisolver.trans_rdm1(mc.ci[ci_id], mc.ci[ci_id], mc.ncas, mc.nelecas)
        return t_dm

    for ex in range(N_state):
        D_cas_active    = cas_make_rdm(ex)
        D_cas_full      = MO2AO.restore_full_space(D_cas_active, Trans)
        D_cas_AO        = MO2AO.run(D_cas_full, mf_hf.mo_coeff)
        np.save(f"{master_dir}/Data/{system}/Density/CASCI_{ex}_{dist}", D_cas_full)
    print("CASCI RDM Done")

    # CASCI Dipole
    dip_frozen      = Dipole.gen_core_dipole(classic_object = PySCFTranspiler(mf_hf, active_o, active_e))
    def makedip(ci_id1, ci_id2):
        t_dm1       = mc.fcisolver.trans_rdm1(mc.ci[ci_id1], mc.ci[ci_id2], mc.ncas, mc.nelecas)
        orbcas      = mc_mo[:,mc.ncore:mc.ncore+mc.ncas]
        t_dm1_ao    = reduce(np.dot, (orbcas, t_dm1, orbcas.T))
        return np.einsum('xij,ji->x', dip_ints, t_dm1_ao)

    for i in range(N_state):
        for j in range(N_state):
            dip                         = makedip(i,j)
            dipole_CAS[ind, i, j, :]    = dip + (dip_frozen if i == j else 0.0)
    print("CASCI Dipole Done")

    np.save(f"{master_dir}/Data/{system}/Energy/{dist}_CAS", energy_CAS[ind])
    np.save(f"{master_dir}/Data/{system}/Dipole/{dist}_CAS", dipole_CAS[ind])


>>>>>> 1.0 AA Start
=====================< PySCF Transpiler >=====================
      Multiplicity : 1
 # Spatial Orbital : 5 (Active Space Applied)
       # Electrons : Alpha(1) | Beta(1)
            # Atom : 2
            Core E : -6.7083 Hartree

WARN: Mulitple states found in CASCI solver. First state is used to compute the Fock matrix and natural orbitals in active space.

CASCI state   0  E = -7.87285340311942  E(CI) = -1.16456034941256  S^2 = 0.0000000
CASCI state   1  E = -7.72009696580792  E(CI) = -1.01180391210106  S^2 = 0.0000000
CASCI state   2  E = -7.67339408642108  E(CI) = -0.965101032714222  S^2 = 0.0000000
CASCI state   3  E = -7.67339408642108  E(CI) = -0.965101032714222  S^2 = 0.0000000
CASCI RDM Done
=====================< PySCF Transpiler >=====================
      Multiplicity : 1
 # Spatial Orbital : 5 (Active Space Applied)
       # Electrons : Alpha(1) | Beta(1)
            # Atom : 2
            Core E : -6.7083 Hartree
CASCI Dipole Done

>>>>>> 1.1 AA S

## $H_2O$

In [31]:
###===================================================<<< Settings

system      = 'H2O'
dist_lst    = np.around(np.arange(0.7, 1.61, 0.1), 3)
N_state     = 3
active_e    = (4, 4)
active_o    = 6

###===================================================<<< Calculation

energy_CAS  = np.zeros((len(dist_lst), N_state))
dipole_CAS  = np.zeros((len(dist_lst), N_state, N_state, 3))

for ind, dist in enumerate(dist_lst):
    print(f"\n>>>>>> {dist} AA Start")
    # HF
    mol, mf_chk         = scf.chkfile.load_scf(f"{master_dir}/Data/{system}/Chkfile/scf_{dist}")
    mf_hf               = scf.RHF(mol)
    mf_hf.__dict__.update(mf_chk)
    Trans               = PySCFTranspiler(mf_hf, active_o, active_e)
    dip_ints            = mol.intor('cint1e_r_sph', comp=3)

    # CASCI
    mc_mo               = mf_hf.mo_coeff
    mc                  = mcscf.CASCI(mf_hf, active_o, active_e)
    mc.fcisolver        = fci.direct_spin0.FCI(mol)
    mc.fcisolver.nroots = N_state
    mc.kernel(mc_mo)
    energy_CAS[ind]     = mc.e_tot

    # CASCI RDM & DC
    def cas_make_rdm(ci_id):
        t_dm = mc.fcisolver.trans_rdm1(mc.ci[ci_id], mc.ci[ci_id], mc.ncas, mc.nelecas)
        return t_dm

    for ex in range(N_state):
        D_cas_active    = cas_make_rdm(ex)
        D_cas_full      = MO2AO.restore_full_space(D_cas_active, Trans)
        D_cas_AO        = MO2AO.run(D_cas_full, mf_hf.mo_coeff)
        np.save(f"{master_dir}/Data/{system}/Density/CASCI_{ex}_{dist}", D_cas_full)
    print("CASCI RDM Done")

    # CASCI Dipole
    dip_frozen      = Dipole.gen_core_dipole(classic_object = PySCFTranspiler(mf_hf, active_o, active_e))
    def makedip(ci_id1, ci_id2):
        t_dm1       = mc.fcisolver.trans_rdm1(mc.ci[ci_id1], mc.ci[ci_id2], mc.ncas, mc.nelecas)
        orbcas      = mc_mo[:,mc.ncore:mc.ncore+mc.ncas]
        t_dm1_ao    = reduce(np.dot, (orbcas, t_dm1, orbcas.T))
        return np.einsum('xij,ji->x', dip_ints, t_dm1_ao)

    for i in range(N_state):
        for j in range(N_state):
            dip                         = makedip(i,j)
            dipole_CAS[ind, i, j, :]    = dip + (dip_frozen if i == j else 0.0)
    print("CASCI Dipole Done")

    np.save(f"{master_dir}/Data/{system}/Energy/{dist}_CAS", energy_CAS[ind])
    np.save(f"{master_dir}/Data/{system}/Dipole/{dist}_CAS", dipole_CAS[ind])


>>>>>> 0.7 AA Start
=====================< PySCF Transpiler >=====================
      Multiplicity : 1
 # Spatial Orbital : 6 (Active Space Applied)
       # Electrons : Alpha(4) | Beta(4)
            # Atom : 3
            Core E : -48.9011 Hartree

WARN: Mulitple states found in CASCI solver. First state is used to compute the Fock matrix and natural orbitals in active space.

CASCI state   0  E = -74.6435750278990  E(CI) = -25.7424673938754  S^2 = 0.0000000
CASCI state   1  E = -73.8455180377021  E(CI) = -24.9444104036785  S^2 = 0.0000000
CASCI state   2  E = -73.7711237102156  E(CI) = -24.8700160761920  S^2 = 0.0000000
CASCI RDM Done
=====================< PySCF Transpiler >=====================
      Multiplicity : 1
 # Spatial Orbital : 6 (Active Space Applied)
       # Electrons : Alpha(4) | Beta(4)
            # Atom : 3
            Core E : -48.9011 Hartree
CASCI Dipole Done

>>>>>> 0.8 AA Start
=====================< PySCF Transpiler >=====================
      Multiplic

## $H_2O$ Small

In [38]:
###===================================================<<< Settings

system      = 'H2O_Small'
dist_lst    = np.around(np.arange(0.7, 1.61, 0.1), 3)
N_state     = 3
active_e    = (4, 4)
active_o    = 5

###===================================================<<< Calculation

energy_CAS  = np.zeros((len(dist_lst), N_state))
dipole_CAS  = np.zeros((len(dist_lst), N_state, N_state, 3))

for ind, dist in enumerate(dist_lst):
    print(f"\n>>>>>> {dist} AA Start")
    # HF
    mol, mf_chk         = scf.chkfile.load_scf(f"{master_dir}/Data/{system}/Chkfile/scf_{dist}")
    mf_hf               = scf.RHF(mol)
    mf_hf.__dict__.update(mf_chk)
    Trans               = PySCFTranspiler(mf_hf, active_o, active_e)
    dip_ints            = mol.intor('cint1e_r_sph', comp=3)

    # CASCI
    mc_mo               = mf_hf.mo_coeff
    mc                  = mcscf.CASCI(mf_hf, active_o, active_e)
    mc.fcisolver        = fci.direct_spin0.FCI(mol)
    mc.fcisolver.nroots = N_state
    mc.kernel(mc_mo)
    energy_CAS[ind]     = mc.e_tot

    # CASCI RDM & DC
    def cas_make_rdm(ci_id):
        t_dm = mc.fcisolver.trans_rdm1(mc.ci[ci_id], mc.ci[ci_id], mc.ncas, mc.nelecas)
        return t_dm

    for ex in range(N_state):
        D_cas_active    = cas_make_rdm(ex)
        D_cas_full      = MO2AO.restore_full_space(D_cas_active, Trans)
        D_cas_AO        = MO2AO.run(D_cas_full, mf_hf.mo_coeff)
        np.save(f"{master_dir}/Data/{system}/Density/CASCI_{ex}_{dist}", D_cas_full)
    print("CASCI RDM Done")

    # CASCI Dipole
    dip_frozen      = Dipole.gen_core_dipole(classic_object = PySCFTranspiler(mf_hf, active_o, active_e))
    def makedip(ci_id1, ci_id2):
        t_dm1       = mc.fcisolver.trans_rdm1(mc.ci[ci_id1], mc.ci[ci_id2], mc.ncas, mc.nelecas)
        orbcas      = mc_mo[:,mc.ncore:mc.ncore+mc.ncas]
        t_dm1_ao    = reduce(np.dot, (orbcas, t_dm1, orbcas.T))
        return np.einsum('xij,ji->x', dip_ints, t_dm1_ao)

    for i in range(N_state):
        for j in range(N_state):
            dip                         = makedip(i,j)
            dipole_CAS[ind, i, j, :]    = dip + (dip_frozen if i == j else 0.0)
    print("CASCI Dipole Done")

    np.save(f"{master_dir}/Data/{system}/Energy/{dist}_CAS", energy_CAS[ind])
    np.save(f"{master_dir}/Data/{system}/Dipole/{dist}_CAS", dipole_CAS[ind])


>>>>>> 0.7 AA Start
=====================< PySCF Transpiler >=====================
      Multiplicity : 1
 # Spatial Orbital : 5 (Active Space Applied)
       # Electrons : Alpha(4) | Beta(4)
            # Atom : 3
            Core E : -48.9011 Hartree

WARN: Mulitple states found in CASCI solver. First state is used to compute the Fock matrix and natural orbitals in active space.

CASCI state   0  E = -74.6262184606958  E(CI) = -25.7251108266722  S^2 = 0.0000000
CASCI state   1  E = -73.8123489927506  E(CI) = -24.9112413587270  S^2 = 0.0000000
CASCI state   2  E = -73.7262540229795  E(CI) = -24.8251463889560  S^2 = 0.0000000


CASCI RDM Done
=====================< PySCF Transpiler >=====================
      Multiplicity : 1
 # Spatial Orbital : 5 (Active Space Applied)
       # Electrons : Alpha(4) | Beta(4)
            # Atom : 3
            Core E : -48.9011 Hartree
CASCI Dipole Done

>>>>>> 0.8 AA Start
=====================< PySCF Transpiler >=====================
      Multiplicity : 1
 # Spatial Orbital : 5 (Active Space Applied)
       # Electrons : Alpha(4) | Beta(4)
            # Atom : 3
            Core E : -50.0948 Hartree

WARN: Mulitple states found in CASCI solver. First state is used to compute the Fock matrix and natural orbitals in active space.

CASCI state   0  E = -74.8599916193103  E(CI) = -24.7651795090529  S^2 = 0.0000000
CASCI state   1  E = -74.1874636637727  E(CI) = -24.0926515535153  S^2 = 0.0000000
CASCI state   2  E = -74.0739032560597  E(CI) = -23.9790911458022  S^2 = 0.0000000
CASCI RDM Done
=====================< PySCF Transpiler >=====================
      Multiplicity : 

## $HeH^+$

In [33]:
###===================================================<<< Settings

system      = 'HeH'
dist_lst    = np.around(np.arange(0.5, 2.01, 0.1), 3)
N_state     = 3
active_e    = (1, 1)
active_o    = 2

###===================================================<<< Calculation

energy_CAS  = np.zeros((len(dist_lst), N_state))
dipole_CAS  = np.zeros((len(dist_lst), N_state, N_state, 3))

for ind, dist in enumerate(dist_lst):
    print(f"\n>>>>>> {dist} AA Start")
    # HF
    mol, mf_chk         = scf.chkfile.load_scf(f"{master_dir}/Data/{system}/Chkfile/scf_{dist}")
    mf_hf               = scf.RHF(mol)
    mf_hf.__dict__.update(mf_chk)
    Trans               = PySCFTranspiler(mf_hf, active_o, active_e)
    dip_ints            = mol.intor('cint1e_r_sph', comp=3)

    # CASCI
    mc_mo               = mf_hf.mo_coeff
    mc                  = mcscf.CASCI(mf_hf, active_o, active_e)
    mc.fcisolver        = fci.direct_spin0.FCI(mol)
    mc.fcisolver.nroots = N_state
    mc.kernel(mc_mo)
    energy_CAS[ind]     = mc.e_tot

    # CASCI RDM & DC
    def cas_make_rdm(ci_id):
        t_dm = mc.fcisolver.trans_rdm1(mc.ci[ci_id], mc.ci[ci_id], mc.ncas, mc.nelecas)
        return t_dm

    for ex in range(N_state):
        D_cas_active    = cas_make_rdm(ex)
        D_cas_full      = MO2AO.restore_full_space(D_cas_active, Trans)
        D_cas_AO        = MO2AO.run(D_cas_full, mf_hf.mo_coeff)
        np.save(f"{master_dir}/Data/{system}/Density/CASCI_{ex}_{dist}", D_cas_full)
    print("CASCI RDM Done")

    # CASCI Dipole
    dip_frozen      = Dipole.gen_core_dipole(classic_object = PySCFTranspiler(mf_hf, active_o, active_e))
    def makedip(ci_id1, ci_id2):
        t_dm1       = mc.fcisolver.trans_rdm1(mc.ci[ci_id1], mc.ci[ci_id2], mc.ncas, mc.nelecas)
        orbcas      = mc_mo[:,mc.ncore:mc.ncore+mc.ncas]
        t_dm1_ao    = reduce(np.dot, (orbcas, t_dm1, orbcas.T))
        return np.einsum('xij,ji->x', dip_ints, t_dm1_ao)

    for i in range(N_state):
        for j in range(N_state):
            dip                         = makedip(i,j)
            dipole_CAS[ind, i, j, :]    = dip + (dip_frozen if i == j else 0.0)
    print("CASCI Dipole Done")

    np.save(f"{master_dir}/Data/{system}/Energy/{dist}_CAS", energy_CAS[ind])
    np.save(f"{master_dir}/Data/{system}/Dipole/{dist}_CAS", dipole_CAS[ind])


>>>>>> 0.5 AA Start
=====================< PySCF Transpiler >=====================
      Multiplicity : 1
 # Spatial Orbital : 2 (Active Space Applied)
       # Electrons : Alpha(1) | Beta(1)
            # Atom : 2
            Core E : 2.1167 Hartree

WARN: Mulitple states found in CASCI solver. First state is used to compute the Fock matrix and natural orbitals in active space.

CASCI state   0  E = -2.64071459048826  E(CI) = -4.75742343416826  S^2 = 0.0000000
CASCI state   1  E = -1.08644257798160  E(CI) = -3.20315142166160  S^2 = 0.0000000
CASCI state   2  E = 0.457085856709396  E(CI) = -1.65962298697060  S^2 = 0.0000000
CASCI RDM Done
=====================< PySCF Transpiler >=====================
      Multiplicity : 1
 # Spatial Orbital : 2 (Active Space Applied)
       # Electrons : Alpha(1) | Beta(1)
            # Atom : 2
            Core E : 2.1167 Hartree
CASCI Dipole Done

>>>>>> 0.6 AA Start
=====================< PySCF Transpiler >=====================
      Multiplicity 

## $BeH_2$

In [34]:
###===================================================<<< Settings

system      = 'BeH2'
dist_lst    = np.around(np.arange(0.5, 1.61, 0.1), 3)
N_state     = 3
active_e    = (1, 1)
active_o    = 5

###===================================================<<< Calculation

energy_CAS  = np.zeros((len(dist_lst), N_state))
dipole_CAS  = np.zeros((len(dist_lst), N_state, N_state, 3))

for ind, dist in enumerate(dist_lst):
    print(f"\n>>>>>> {dist} AA Start")
    # HF
    mol, mf_chk         = scf.chkfile.load_scf(f"{master_dir}/Data/{system}/Chkfile/scf_{dist}")
    mf_hf               = scf.RHF(mol)
    mf_hf.__dict__.update(mf_chk)
    Trans               = PySCFTranspiler(mf_hf, active_o, active_e)
    dip_ints            = mol.intor('cint1e_r_sph', comp=3)

    # CASCI
    mc_mo               = mf_hf.mo_coeff
    mc                  = mcscf.CASCI(mf_hf, active_o, active_e)
    mc.fcisolver        = fci.direct_spin0.FCI(mol)
    mc.fcisolver.nroots = N_state
    mc.kernel(mc_mo)
    energy_CAS[ind]     = mc.e_tot

    # CASCI RDM & DC
    def cas_make_rdm(ci_id):
        t_dm = mc.fcisolver.trans_rdm1(mc.ci[ci_id], mc.ci[ci_id], mc.ncas, mc.nelecas)
        return t_dm

    for ex in range(N_state):
        D_cas_active    = cas_make_rdm(ex)
        D_cas_full      = MO2AO.restore_full_space(D_cas_active, Trans)
        D_cas_AO        = MO2AO.run(D_cas_full, mf_hf.mo_coeff)
        np.save(f"{master_dir}/Data/{system}/Density/CASCI_{ex}_{dist}", D_cas_full)
    print("CASCI RDM Done")

    # CASCI Dipole
    dip_frozen      = Dipole.gen_core_dipole(classic_object = PySCFTranspiler(mf_hf, active_o, active_e))
    def makedip(ci_id1, ci_id2):
        t_dm1       = mc.fcisolver.trans_rdm1(mc.ci[ci_id1], mc.ci[ci_id2], mc.ncas, mc.nelecas)
        orbcas      = mc_mo[:,mc.ncore:mc.ncore+mc.ncas]
        t_dm1_ao    = reduce(np.dot, (orbcas, t_dm1, orbcas.T))
        return np.einsum('xij,ji->x', dip_ints, t_dm1_ao)

    for i in range(N_state):
        for j in range(N_state):
            dip                         = makedip(i,j)
            dipole_CAS[ind, i, j, :]    = dip + (dip_frozen if i == j else 0.0)
    print("CASCI Dipole Done")

    np.save(f"{master_dir}/Data/{system}/Energy/{dist}_CAS", energy_CAS[ind])
    np.save(f"{master_dir}/Data/{system}/Dipole/{dist}_CAS", dipole_CAS[ind])


>>>>>> 0.5 AA Start
=====================< PySCF Transpiler >=====================
      Multiplicity : 1
 # Spatial Orbital : 5 (Active Space Applied)
       # Electrons : Alpha(1) | Beta(1)
            # Atom : 3
            Core E : -12.1950 Hartree

WARN: Mulitple states found in CASCI solver. First state is used to compute the Fock matrix and natural orbitals in active space.

CASCI state   0  E = -13.6699200753536  E(CI) = -1.47489477294169  S^2 = 0.0000000
CASCI state   1  E = -13.3140544733319  E(CI) = -1.11902917092004  S^2 = 0.0000000
CASCI state   2  E = -13.3140544733319  E(CI) = -1.11902917092004  S^2 = 0.0000000
CASCI RDM Done
=====================< PySCF Transpiler >=====================
      Multiplicity : 1
 # Spatial Orbital : 5 (Active Space Applied)
       # Electrons : Alpha(1) | Beta(1)
            # Atom : 3
            Core E : -12.1950 Hartree
CASCI Dipole Done

>>>>>> 0.6 AA Start
=====================< PySCF Transpiler >=====================
      Multiplic

## $NH_3$

In [35]:
###===================================================<<< Settings

system      = 'NH3'
dist_lst    = np.around(np.arange(0.5, 2.01, 0.1), 3)
N_state     = 3
active_e    = (1, 1)
active_o    = 4

###===================================================<<< Calculation

energy_CAS  = np.zeros((len(dist_lst), N_state))
dipole_CAS  = np.zeros((len(dist_lst), N_state, N_state, 3))

for ind, dist in enumerate(dist_lst):
    print(f"\n>>>>>> {dist} AA Start")
    # HF
    mol, mf_chk         = scf.chkfile.load_scf(f"{master_dir}/Data/{system}/Chkfile/scf_{dist}")
    mf_hf               = scf.RHF(mol)
    mf_hf.__dict__.update(mf_chk)
    Trans               = PySCFTranspiler(mf_hf, active_o, active_e)
    dip_ints            = mol.intor('cint1e_r_sph', comp=3)

    # CASCI
    mc_mo               = mf_hf.mo_coeff
    mc                  = mcscf.CASCI(mf_hf, active_o, active_e)
    mc.fcisolver        = fci.direct_spin0.FCI(mol)
    mc.fcisolver.nroots = N_state
    mc.kernel(mc_mo)
    energy_CAS[ind]     = mc.e_tot

    # CASCI RDM & DC
    def cas_make_rdm(ci_id):
        t_dm = mc.fcisolver.trans_rdm1(mc.ci[ci_id], mc.ci[ci_id], mc.ncas, mc.nelecas)
        return t_dm

    for ex in range(N_state):
        D_cas_active    = cas_make_rdm(ex)
        D_cas_full      = MO2AO.restore_full_space(D_cas_active, Trans)
        D_cas_AO        = MO2AO.run(D_cas_full, mf_hf.mo_coeff)
        np.save(f"{master_dir}/Data/{system}/Density/CASCI_{ex}_{dist}", D_cas_full)
    print("CASCI RDM Done")

    # CASCI Dipole
    dip_frozen      = Dipole.gen_core_dipole(classic_object = PySCFTranspiler(mf_hf, active_o, active_e))
    def makedip(ci_id1, ci_id2):
        t_dm1       = mc.fcisolver.trans_rdm1(mc.ci[ci_id1], mc.ci[ci_id2], mc.ncas, mc.nelecas)
        orbcas      = mc_mo[:,mc.ncore:mc.ncore+mc.ncas]
        t_dm1_ao    = reduce(np.dot, (orbcas, t_dm1, orbcas.T))
        return np.einsum('xij,ji->x', dip_ints, t_dm1_ao)

    for i in range(N_state):
        for j in range(N_state):
            dip                         = makedip(i,j)
            dipole_CAS[ind, i, j, :]    = dip + (dip_frozen if i == j else 0.0)
    print("CASCI Dipole Done")

    np.save(f"{master_dir}/Data/{system}/Energy/{dist}_CAS", energy_CAS[ind])
    np.save(f"{master_dir}/Data/{system}/Dipole/{dist}_CAS", dipole_CAS[ind])


>>>>>> 0.5 AA Start
=====================< PySCF Transpiler >=====================
      Multiplicity : 1
 # Spatial Orbital : 4 (Active Space Applied)
       # Electrons : Alpha(1) | Beta(1)
            # Atom : 4
            Core E : -50.4344 Hartree

WARN: Mulitple states found in CASCI solver. First state is used to compute the Fock matrix and natural orbitals in active space.

CASCI state   0  E = -52.4371625554672  E(CI) = -2.00273220633919  S^2 = 0.0000000
CASCI state   1  E = -50.9612928329273  E(CI) = -0.526862483799313  S^2 = 0.0000000
CASCI state   2  E = -50.8386578650990  E(CI) = -0.404227515970923  S^2 = 0.0000000
CASCI RDM Done
=====================< PySCF Transpiler >=====================
      Multiplicity : 1
 # Spatial Orbital : 4 (Active Space Applied)
       # Electrons : Alpha(1) | Beta(1)
            # Atom : 4
            Core E : -50.4344 Hartree
CASCI Dipole Done

>>>>>> 0.6 AA Start
=====================< PySCF Transpiler >=====================
      Multipl

## $HF$

In [36]:
###===================================================<<< Settings

system      = 'HF'
dist_lst    = np.around(np.arange(0.5, 2.01, 0.1), 3)
N_state     = 3
active_e    = (4, 4)
active_o    = 5

###===================================================<<< Calculation

energy_CAS  = np.zeros((len(dist_lst), N_state))
dipole_CAS  = np.zeros((len(dist_lst), N_state, N_state, 3))

for ind, dist in enumerate(dist_lst):
    print(f"\n>>>>>> {dist} AA Start")
    # HF
    mol, mf_chk         = scf.chkfile.load_scf(f"{master_dir}/Data/{system}/Chkfile/scf_{dist}")
    mf_hf               = scf.RHF(mol)
    mf_hf.__dict__.update(mf_chk)
    Trans               = PySCFTranspiler(mf_hf, active_o, active_e)
    dip_ints            = mol.intor('cint1e_r_sph', comp=3)

    # CASCI
    mc_mo               = mf_hf.mo_coeff
    mc                  = mcscf.CASCI(mf_hf, active_o, active_e)
    mc.fcisolver        = fci.direct_spin0.FCI(mol)
    mc.fcisolver.nroots = N_state
    mc.kernel(mc_mo)
    energy_CAS[ind]     = mc.e_tot

    # CASCI RDM & DC
    def cas_make_rdm(ci_id):
        t_dm = mc.fcisolver.trans_rdm1(mc.ci[ci_id], mc.ci[ci_id], mc.ncas, mc.nelecas)
        return t_dm

    for ex in range(N_state):
        D_cas_active    = cas_make_rdm(ex)
        D_cas_full      = MO2AO.restore_full_space(D_cas_active, Trans)
        D_cas_AO        = MO2AO.run(D_cas_full, mf_hf.mo_coeff)
        np.save(f"{master_dir}/Data/{system}/Density/CASCI_{ex}_{dist}", D_cas_full)
    print("CASCI RDM Done")

    # CASCI Dipole
    dip_frozen      = Dipole.gen_core_dipole(classic_object = PySCFTranspiler(mf_hf, active_o, active_e))
    def makedip(ci_id1, ci_id2):
        t_dm1       = mc.fcisolver.trans_rdm1(mc.ci[ci_id1], mc.ci[ci_id2], mc.ncas, mc.nelecas)
        orbcas      = mc_mo[:,mc.ncore:mc.ncore+mc.ncas]
        t_dm1_ao    = reduce(np.dot, (orbcas, t_dm1, orbcas.T))
        return np.einsum('xij,ji->x', dip_ints, t_dm1_ao)

    for i in range(N_state):
        for j in range(N_state):
            dip                         = makedip(i,j)
            dipole_CAS[ind, i, j, :]    = dip + (dip_frozen if i == j else 0.0)
    print("CASCI Dipole Done")

    np.save(f"{master_dir}/Data/{system}/Energy/{dist}_CAS", energy_CAS[ind])
    np.save(f"{master_dir}/Data/{system}/Dipole/{dist}_CAS", dipole_CAS[ind])


>>>>>> 0.5 AA Start
=====================< PySCF Transpiler >=====================
      Multiplicity : 1
 # Spatial Orbital : 5 (Active Space Applied)
       # Electrons : Alpha(4) | Beta(4)
            # Atom : 2
            Core E : -67.2428 Hartree

WARN: Mulitple states found in CASCI solver. First state is used to compute the Fock matrix and natural orbitals in active space.

CASCI state   0  E = -97.7138223974666  E(CI) = -30.4709952760310  S^2 = 0.0000000
CASCI state   1  E = -96.7068179864108  E(CI) = -29.4639908649752  S^2 = 0.0000000
CASCI state   2  E = -96.7068179864107  E(CI) = -29.4639908649752  S^2 = 0.0000000
CASCI RDM Done
=====================< PySCF Transpiler >=====================
      Multiplicity : 1
 # Spatial Orbital : 5 (Active Space Applied)
       # Electrons : Alpha(4) | Beta(4)
            # Atom : 2
            Core E : -67.2428 Hartree
CASCI Dipole Done

>>>>>> 0.6 AA Start
=====================< PySCF Transpiler >=====================
      Multiplic

## $H_2$ Origin Shift

In [3]:
###===================================================<<< Settings

system      = 'H2_Origin_Shift'
dist_lst    = np.around(np.arange(0.0, 1.01, 0.1), 3)
N_state     = 3
active_e    = (1, 1)
active_o    = 2

###===================================================<<< Calculation

energy_CAS  = np.zeros((len(dist_lst), N_state))
dipole_CAS  = np.zeros((len(dist_lst), N_state, N_state, 3))

for ind, dist in enumerate(dist_lst):
    print(f"\n>>>>>> {dist} AA Start")
    # HF
    mol, mf_chk         = scf.chkfile.load_scf(f"{master_dir}/Data/{system}/Chkfile/scf_{dist}")
    mf_hf               = scf.RHF(mol)
    mf_hf.__dict__.update(mf_chk)
    Trans               = PySCFTranspiler(mf_hf, active_o, active_e)
    dip_ints            = mol.intor('cint1e_r_sph', comp=3)

    # CASCI
    mc_mo               = mf_hf.mo_coeff
    mc                  = mcscf.CASCI(mf_hf, active_o, active_e)
    mc.fcisolver        = fci.direct_spin0.FCI(mol)
    mc.fcisolver.nroots = N_state
    mc.kernel(mc_mo)
    energy_CAS[ind]     = mc.e_tot

    # CASCI RDM & DC
    def cas_make_rdm(ci_id):
        t_dm = mc.fcisolver.trans_rdm1(mc.ci[ci_id], mc.ci[ci_id], mc.ncas, mc.nelecas)
        return t_dm

    for ex in range(N_state):
        D_cas_active    = cas_make_rdm(ex)
        D_cas_full      = MO2AO.restore_full_space(D_cas_active, Trans)
        D_cas_AO        = MO2AO.run(D_cas_full, mf_hf.mo_coeff)
        np.save(f"{master_dir}/Data/{system}/Density/CASCI_{ex}_{dist}", D_cas_full)
    print("CASCI RDM Done")

    # CASCI Dipole
    dip_frozen      = Dipole.gen_core_dipole(classic_object = PySCFTranspiler(mf_hf, active_o, active_e))
    def makedip(ci_id1, ci_id2):
        t_dm1       = mc.fcisolver.trans_rdm1(mc.ci[ci_id1], mc.ci[ci_id2], mc.ncas, mc.nelecas)
        orbcas      = mc_mo[:,mc.ncore:mc.ncore+mc.ncas]
        t_dm1_ao    = reduce(np.dot, (orbcas, t_dm1, orbcas.T))
        return np.einsum('xij,ji->x', dip_ints, t_dm1_ao)

    for i in range(N_state):
        for j in range(N_state):
            dip                         = makedip(i,j)
            dipole_CAS[ind, i, j, :]    = dip + (dip_frozen if i == j else 0.0)
    print("CASCI Dipole Done")

    np.save(f"{master_dir}/Data/{system}/Energy/{dist}_CAS", energy_CAS[ind])
    np.save(f"{master_dir}/Data/{system}/Dipole/{dist}_CAS", dipole_CAS[ind])


>>>>>> 0.0 AA Start
=====================< PySCF Transpiler >=====================
      Multiplicity : 1
 # Spatial Orbital : 2 (Active Space Applied)
       # Electrons : Alpha(1) | Beta(1)
            # Atom : 2
            Core E : 0.5292 Hartree

WARN: Mulitple states found in CASCI solver. First state is used to compute the Fock matrix and natural orbitals in active space.

CASCI state   0  E = -1.10115033023262  E(CI) = -1.63032754115262  S^2 = 0.0000000
CASCI state   1  E = -0.352290626064627  E(CI) = -0.881467836984627  S^2 = 0.0000000
CASCI state   2  E = 0.0390476313650900  E(CI) = -0.490129579554910  S^2 = 0.0000000
CASCI RDM Done
=====================< PySCF Transpiler >=====================
      Multiplicity : 1
 # Spatial Orbital : 2 (Active Space Applied)
       # Electrons : Alpha(1) | Beta(1)
            # Atom : 2
            Core E : 0.5292 Hartree
CASCI Dipole Done

>>>>>> 0.1 AA Start
=====================< PySCF Transpiler >=====================
      Multiplic

## $LiH$ Origin Shift

In [4]:
###===================================================<<< Settings

system      = 'LiH_Origin_Shift'
dist_lst    = np.around(np.arange(0.0, 2.01, 0.2), 3)
N_state     = 4
active_e    = (1, 1)
active_o    = 5

###===================================================<<< Calculation

energy_CAS  = np.zeros((len(dist_lst), N_state))
dipole_CAS  = np.zeros((len(dist_lst), N_state, N_state, 3))

for ind, dist in enumerate(dist_lst):
    print(f"\n>>>>>> {dist} AA Start")
    # HF
    mol, mf_chk         = scf.chkfile.load_scf(f"{master_dir}/Data/{system}/Chkfile/scf_{dist}")
    mf_hf               = scf.RHF(mol)
    mf_hf.__dict__.update(mf_chk)
    Trans               = PySCFTranspiler(mf_hf, active_o, active_e)
    dip_ints            = mol.intor('cint1e_r_sph', comp=3)

    # CASCI
    mc_mo               = mf_hf.mo_coeff
    mc                  = mcscf.CASCI(mf_hf, active_o, active_e)
    mc.fcisolver        = fci.direct_spin0.FCI(mol)
    mc.fcisolver.nroots = N_state
    mc.kernel(mc_mo)
    energy_CAS[ind]     = mc.e_tot

    # CASCI RDM & DC
    def cas_make_rdm(ci_id):
        t_dm = mc.fcisolver.trans_rdm1(mc.ci[ci_id], mc.ci[ci_id], mc.ncas, mc.nelecas)
        return t_dm

    for ex in range(N_state):
        D_cas_active    = cas_make_rdm(ex)
        D_cas_full      = MO2AO.restore_full_space(D_cas_active, Trans)
        D_cas_AO        = MO2AO.run(D_cas_full, mf_hf.mo_coeff)
        np.save(f"{master_dir}/Data/{system}/Density/CASCI_{ex}_{dist}", D_cas_full)
    print("CASCI RDM Done")

    # CASCI Dipole
    dip_frozen      = Dipole.gen_core_dipole(classic_object = PySCFTranspiler(mf_hf, active_o, active_e))
    def makedip(ci_id1, ci_id2):
        t_dm1       = mc.fcisolver.trans_rdm1(mc.ci[ci_id1], mc.ci[ci_id2], mc.ncas, mc.nelecas)
        orbcas      = mc_mo[:,mc.ncore:mc.ncore+mc.ncas]
        t_dm1_ao    = reduce(np.dot, (orbcas, t_dm1, orbcas.T))
        return np.einsum('xij,ji->x', dip_ints, t_dm1_ao)

    for i in range(N_state):
        for j in range(N_state):
            dip                         = makedip(i,j)
            dipole_CAS[ind, i, j, :]    = dip + (dip_frozen if i == j else 0.0)
    print("CASCI Dipole Done")

    np.save(f"{master_dir}/Data/{system}/Energy/{dist}_CAS", energy_CAS[ind])
    np.save(f"{master_dir}/Data/{system}/Dipole/{dist}_CAS", dipole_CAS[ind])


>>>>>> 0.0 AA Start
=====================< PySCF Transpiler >=====================
      Multiplicity : 1
 # Spatial Orbital : 5 (Active Space Applied)
       # Electrons : Alpha(1) | Beta(1)
            # Atom : 2
            Core E : -6.8704 Hartree

WARN: Mulitple states found in CASCI solver. First state is used to compute the Fock matrix and natural orbitals in active space.

CASCI state   0  E = -7.86082825823076  E(CI) = -0.990413579803615  S^2 = 0.0000000
CASCI state   1  E = -7.75228233252575  E(CI) = -0.881867654098603  S^2 = 0.0000000
CASCI state   2  E = -7.70454384492689  E(CI) = -0.834129166499747  S^2 = 0.0000000
CASCI state   3  E = -7.70454384492689  E(CI) = -0.834129166499746  S^2 = 0.0000000
CASCI RDM Done
=====================< PySCF Transpiler >=====================
      Multiplicity : 1
 # Spatial Orbital : 5 (Active Space Applied)
       # Electrons : Alpha(1) | Beta(1)
            # Atom : 2
            Core E : -6.8704 Hartree
CASCI Dipole Done

>>>>>> 0.2 AA